In [1]:
# This test program aims to output a list of n_mc coordinates within a sample.
# This program will use ray-tracing and the odd-even rule to determine whether a random coordinate is within a sample.

using MeshIO
using FileIO
using BenchmarkTools
using StaticArrays
using GeometryBasics
include("Modules/sampling.jl")
using .sampling

In [2]:
# Setting the desired number of MC sample points.

const n_mc = 10

10

In [3]:
# Retrieving the vertices and indices of the triangular mesh of the sample surface from the .stl file.

# Dummy multiple-crystal sample comprising 7 icospheres with 80 faces each.
stl = load("STL_FileExamples/7_Icospheres80.stl")
vertices = GeometryBasics.coordinates(stl)
indices = GeometryBasics.faces(stl)
# Extracting the number of triangular faces used in the mesh.
const n_faces = length(indices)

560

In [4]:
# Storing the coordinates into a vector of length n_mc.

mc_coords = Vector{SVector{3, Float32}}(undef, n_mc)

10-element Vector{SVector{3, Float32}}:
 [0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0]

In [5]:
# Calculating and storing the vectors parallel to each face, e2 = V2 - V1 and e3 = V3 - V1, and V1s specifically in preparation for the Moller-Trumbore Algorithm.
# The vertices of the triangular faces are labelled V1, V2, V3.

v1s, e2s, e3s = sampling.ve_calc(vertices, indices)

(SVector{3, Float32}[[-0.58778524, 0.809017, 0.0], [0.0, -1.0, 0.0], [0.58778524, 0.809017, 0.0], [-0.95105654, 0.309017, 0.0], [0.58778524, -0.809017, 0.0], [-0.4253254, -0.309017, 0.8506508], [0.4253254, 0.309017, -0.8506508], [-0.68819094, 0.5, -0.5257311], [-0.5257311, 0.0, -0.8506508], [-0.68819094, -0.5, -0.5257311]  …  [-0.4253254, -0.309017, -2.1493492], [0.16245985, -0.5, -2.1493492], [0.68819094, -0.5, -2.474269], [0.16245985, -0.5, -2.1493492], [0.5257311, 0.0, -2.1493492], [0.68819094, 0.5, -2.474269], [0.5257311, 0.0, -2.1493492], [0.16245985, 0.5, -2.1493492], [-0.26286554, 0.809017, -2.474269], [0.16245985, 0.5, -2.1493492]], SVector{3, Float32}[[0.58778524, 0.190983, 0.0], [-0.58778524, 0.190983, 0.0], [-0.58778524, 0.190983, 0.0], [0.10040575, -0.309017, 0.5257311], [0.10040569, 0.309017, 0.5257311], [-0.4253254, 0.309017, -0.3249197], [-0.16245985, 0.5, 0.3249197], [0.10040569, 0.309017, 0.5257311], [0.36327124, 0.5, 0.0], [0.5257311, 0.0, -0.3249197]  …  [0.16245985,

In [6]:
# Generating the sample points.

# Calculating the extrema of the axis-aligned bounding box around the sample.
ranges = sampling.aabb_3d(vertices)
sampling.sample!(ranges, e2s, e3s, v1s, mc_coords, n_faces, n_mc)

10-element Vector{SVector{3, Float32}}:
 [0.56146574, 0.0072396914, -3.696]
 [3.2256234, -0.4899119, -0.65174884]
 [0.7425236, -3.3373797, 0.3643379]
 [2.8286738, 0.2841164, 0.52820516]
 [0.5711935, -3.007497, 0.0905966]
 [0.6940449, -2.593371, 0.33810765]
 [0.10946633, 2.6882014, 0.18888128]
 [-2.5175683, 0.04304895, 0.8344667]
 [-0.49324492, -2.8084009, -0.40819183]
 [2.5557654, 0.1545259, -0.056089137]

In [7]:
# Benchmarking this sample point generation function.

@benchmark sampling.sample!(ranges, e2s, e3s, v1s, mc_coords, n_faces, n_mc)

BenchmarkTools.Trial: 7436 samples with 1 evaluation per sample.
 Range (min … max):  162.500 μs …  11.793 ms  ┊ GC (min … max): 0.00% … 94.28%
 Time  (median):     638.000 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   667.296 μs ± 257.601 μs  ┊ GC (mean ± σ):  0.22% ±  1.09%

              ▂▂▄▆▅█▇▆▇██▇███▅▅▃▄▂▃▂▂                            
  ▁▁▁▁▂▃▃▄▅▇▇█████████████████████████▇▇▇▅▆▆▄▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▂ ▅
  162 μs           Histogram: frequency by time         1.35 ms <

 Memory estimate: 9.08 KiB, allocs estimate: 10.